# 01 - Component Tests

Unit/smoke tests for each adapter. Synthetic checks never need weights/video; real-model checks are skipped with a message if files are missing.


## Setup


In [ ]:
import os, sys, pathlib

# --- point these at the real locations on Colab -----------------------------
AI_DIR = os.environ.get("TRAFFIQ_AI_DIR", "/content/traffIQ/ai")
WEIGHTS_DIR = os.environ.get("TRAFFIQ_WEIGHTS_DIR", "")

# /content/implementation is where notebook 00 wrote the modules (or your
# cloned/Mounted repo if you prefer to import from there instead).
sys.path.insert(0, "/content/implementation")
os.environ["TRAFFIQ_AI_DIR"] = str(AI_DIR)

from pipeline.config import Config, colab_config

if WEIGHTS_DIR:
    cfg = colab_config(weights_dir=WEIGHTS_DIR, ai_dir=AI_DIR)
else:
    cfg = Config(ai_dir=AI_DIR)
cfg.validate(require_all=False)
print("ai_dir      :", cfg.ai_dir)
print("ref_repo    :", cfg.ref_repo_dir)
print("plate_weights:", cfg.plate_weights)
print("vehicle_weights:", cfg.vehicle_weights)


## Test 1 - CenterPointTracker (ID persistence / reassign / prune)


In [ ]:
from adapters.tracking_adapter import CenterPointTracker
tracker = CenterPointTracker(35.0)
t1 = tracker.update([(10, 10, 60, 60)])
t2 = tracker.update([(12, 12, 62, 62)])     # near previous -> same ID
t3 = tracker.update([(400, 400, 450, 450)]) # far -> new ID
assert t1[0].id == t2[0].id, (t1, t2)
assert t3[0].id != t2[0].id, (t2, t3)
n_centers = len(tracker.center_points)
_ = tracker.update([])                      # nothing matched -> prune unused
assert len(tracker.center_points) == 0, tracker.center_points
print(f"Test 1 OK: same_id={t2[0].id} new_id={t3[0].id} pruned={n_centers}->{len(tracker.center_points)}")


## Test 2 - LineCrossingSpeedEstimator (synthetic, video time)


In [ ]:
from adapters.tracking_adapter import TrackedBox
from adapters.speed_adapter import LineCrossingSpeedEstimator

# Vehicle crosses line A at frame 100 (t=4.0s) and line B at frame 108 (t=4.32s).
# distance 10 m / 0.32 s -> 31.25 m/s -> 112.5 km/h
speed = LineCrossingSpeedEstimator(cfg)
assert speed.update(TrackedBox(0, 0, 50, 50, 1, 25, cfg.line_a_y), 100, 25.0) is None
ev = speed.update(TrackedBox(0, 0, 50, 50, 1, 25, cfg.line_b_y), 108, 25.0)
assert ev is not None and ev["direction"] == cfg.direction_ab
assert abs(ev["value_kmh"] - 112.5) < 1.0, ev
print(f"Test 2 OK: {ev}")


## Test 3 - VehicleDetector on a sample frame (skips if weights/video missing)


In [ ]:
import cv2, glob
VIDEO = glob.glob("/content/[Hh]ighway*.mp4")
VIDEO += glob.glob("/content/drive/MyDrive/**/[Hh]ighway*.mp4", recursive=True)
sample = None
if pathlib.Path(cfg.vehicle_weights).exists() and VIDEO:
    cap = cv2.VideoCapture(VIDEO[0]); ok, sample = cap.read(); cap.release()
if sample is None:
    print("Test 3 SKIPPED: need object.pt weights + a highway mp4")
else:
    from adapters.vehicle_detector import VehicleDetector
    det = VehicleDetector(cfg)
    vehicles = det.detect(sample)
    assert isinstance(vehicles, list)
    for v in vehicles:
        assert set(v) >= {"bbox", "type", "type_confidence"}
        assert len(v["bbox"]) == 4 and 0.0 <= v["type_confidence"] <= 1.0
    print(f"Test 3 OK: {len(vehicles)} vehicles in first frame in {VIDEO[0]}")


## Test 4 - PlateDetector + PlateRecognizer on a plate crop (skips if weights/video missing)


In [ ]:
if sample is None:
    print("Test 4 SKIPPED (no sample frame)")
else:
    from adapters.plate_detector import PlateDetector
    from adapters.plate_recognition import PlateRecognizerAdapter
    pd_ = PlateDetector(cfg)
    pr_ = PlateRecognizerAdapter(cfg)
    plates = pd_.detect(sample)
    print(f"plates detected: {len(plates)}")
    for p in plates[:3]:
        crop = pd_.crop(sample, p["bbox"])
        rec = pr_.recognize(crop)
        assert "text" in rec and "format_valid" in rec and "confidence" in rec
        print("plate:", p["bbox"], "->", rec)


## Test 5 - JSON contract (exact backend shape)


In [ ]:
from pipeline.schemas import DetectionEvent, VehicleInfo, PlateInfo, SpeedInfo, EventIdFactory, make_timestamp
import json
evt = DetectionEvent(
    event_id=EventIdFactory().next_id(), event_type=cfg.event_type, camera_id=cfg.camera_id,
    timestamp=make_timestamp("2026-09-10T20:45:00", 30.0), local_track_id=42,
    vehicle=VehicleInfo("car", 0.96), plate=PlateInfo("WB12AB1234", 0.91, True),
    speed=SpeedInfo(47.5, True, "NORTH"))
d = evt.to_dict()
expected = {"event_id", "event_type", "camera_id", "timestamp", "local_track_id",
            "vehicle", "plate", "speed"}
assert set(d) == expected, set(d)
assert set(d["vehicle"]) == {"type", "type_confidence"}
assert set(d["plate"]) == {"text", "confidence", "format_valid"}
assert set(d["speed"]) == {"value_kmh", "estimated", "direction"}
assert "Z" not in d["timestamp"]
print("Test 5 OK:\n", json.dumps(d, indent=2))


All tests passed if no assertion failed.
